# 05 Create SFINCS Scenarios

## Stage Contract

Requires:
- Event Catalog scenario rows
- Wflow base models with prepared native instates/restart files
- SFINCS base model domains with native infiltration, roughness, and source points

Produces:
- location-named joint Wflow-SFINCS cluster worklists
- copy-pasteable commands for event-atomic Wflow dynamic handoff -> SFINCS staging -> SFINCS runs
- optional accepted-only local SFINCS staging artifacts for debugging

## Imports and Runtime


In [ ]:
from pathlib import Path
import json
import sys

import pandas as pd
import yaml

notebook_root = Path.cwd().resolve()
location_root = notebook_root.parent
location_name = location_root.name
repo_root = location_root.parents[1]
src_root = repo_root / "src"
if src_root.exists() and str(src_root) not in sys.path:
    sys.path.insert(0, str(src_root))

from study_location import define_location
from sfincs_runs.scenarios import (
    accepted_dynamic_handoff_event_ids,
    audit_inland_coupled_batch_readiness,
    dynamic_handoff_readiness_table,
    stage_inland_coupled_scenarios,
    stage_inland_coupled_scenario_forcing,
)

definition = define_location(location_root / "config.yaml")
runtime_config = definition.config
config = runtime_config
sfincs_config = runtime_config
wflow_config = {"wflow": runtime_config["wflow"]}
runtime_config["wflow"]["domain_set"]["review_required"] = False
wflow_handoff_manifest = runtime_config.get("wflow", {}).get("handoff", {}).get(
    "manifest",
    "data/wflow/domain_set_handoff.yaml",
)

def resolve_location_path(relative_path):
    return location_root / relative_path

pd.set_option("display.max_colwidth", 140)


## Required Inputs


In [ ]:
from wflow_runs.notebook import exists_table
exists_table(location_root, {
    "scenario catalog": "data/event_catalog/catalog/scenario_catalog.csv",
    "probability catalog": "data/event_catalog/catalog/probability_catalog.csv",
    "resilience stress/training catalog": "data/event_catalog/catalog/resilience_stress_training_catalog.csv",
    "event manifest": "data/event_catalog/event_manifest.yaml",
    "wflow handoff manifest": wflow_handoff_manifest,
    "wflow events root": wflow_config["wflow"]["events_root"],
    "sfincs domain manifest": sfincs_config["sfincs_domain_set"]["domain_manifest"],
    "sfincs base model root": sfincs_config["paths"]["base_model_root"],
})


## Scenario Build Plan


In [ ]:
scenario_build = sfincs_config["scenario_build"]
pd.Series({
    "design_scenario": scenario_build["design_scenario"],
    "forcing_variable": scenario_build["forcing_variable"],
    "zsini_mode": scenario_build["zsini_mode"],
    "spinup_hours": scenario_build["timing"]["spinup_hours"],
    "drain_down_hours": scenario_build["timing"]["drain_down_hours"],
    "min_run_hours": scenario_build["timing"]["min_run_hours"],
    "max_run_hours": scenario_build["timing"]["max_run_hours"],
})


## Wflow Event Forcing


In [ ]:
pd.Series({
    "discharge_source": runtime_config["inland_coupling"]["discharge_forcing"]["source"],
    "events_root": wflow_config["wflow"]["events_root"],
    "handoff_manifest": wflow_handoff_manifest,
    "acceptance_artifact": "data/wflow/events/<event_id>/sfincs_discharge.dynamic_handoff.json",
    "sfincs_discharge": "data/wflow/events/<event_id>/sfincs_discharge.nc",
    "zero_rain_control_required": True,
}, name="dynamic_wflow_handoff_contract")


## Coupled Wflow-SFINCS Cluster Pipeline Prep


In [ ]:
# This prepares the cluster worklist and commands for the event-atomic production path:
# Wflow dynamic handoff -> native SFINCS staging -> SFINCS run, per event.
# It does not run Wflow or SFINCS locally.
coupled_pipeline_slurm = "cluster/run_wflow_sfincs_dsai_inland_coupled.slurm"
scenario_root_for_worklists = resolve_location_path(sfincs_config["paths"]["scenarios_root"])
scenario_root_for_worklists.mkdir(parents=True, exist_ok=True)

pipeline_readiness = dynamic_handoff_readiness_table(
    runtime_config,
    location_root,
    catalog_path=resolve_location_path("data/event_catalog/catalog/scenario_catalog.csv"),
    limit=scenario_limit if "scenario_limit" in globals() else None,
)
pipeline_accepted = pipeline_readiness.loc[pipeline_readiness["status"].eq("accepted")].copy()
pipeline_blocked = pipeline_readiness.loc[pipeline_readiness["status"].ne("accepted")].copy()

joint_worklist_csv = scenario_root_for_worklists / f"{location_name}_joint_wflow_sfincs_worklist.csv"
readiness_worklist_csv = scenario_root_for_worklists / f"{location_name}_dynamic_handoff_readiness.csv"
blocked_worklist_csv = scenario_root_for_worklists / f"{location_name}_blocked_dynamic_handoffs.csv"
accepted_worklist_csv = scenario_root_for_worklists / f"{location_name}_accepted_dynamic_handoffs.csv"
pipeline_readiness.to_csv(joint_worklist_csv, index=False)
pipeline_readiness.to_csv(readiness_worklist_csv, index=False)
pipeline_blocked.to_csv(blocked_worklist_csv, index=False)
pipeline_accepted.to_csv(accepted_worklist_csv, index=False)

coupled_array_chunks = 32
coupled_max_concurrent_nodes = 8
# Pick a real catalog event so the debug command below is copy-pasteable.
coupled_debug_event = (
    str(pipeline_readiness["event_id"].iloc[0])
    if not pipeline_readiness.empty
    else "<event_id>"
)

coupled_pipeline_commands = [
    f"SYNC_LOCATION={location_name} cluster/sync_to_dsai.sh --run-inputs-only",
    "mkdir -p runs",
    f"sbatch --array=0-0 --export=ALL,LOCATION={location_name},EVENT_IDS={coupled_debug_event},FORCE_RERUN=1,FORCE_WFLOW=1,OVERWRITE_METEO=1,KEEP_STAGE=1 {coupled_pipeline_slurm}",
    f"sbatch --array=0-{coupled_array_chunks - 1}%{coupled_max_concurrent_nodes} --export=ALL,LOCATION={location_name},WORKLIST=locations/{location_name}/data/sfincs/scenarios/{location_name}_joint_wflow_sfincs_worklist.csv,FORCE_RERUN=1,FORCE_WFLOW=1,OVERWRITE_METEO=1 {coupled_pipeline_slurm}",
]
print("# Submit this joint pipeline after syncing inputs to the cluster:")
for command in coupled_pipeline_commands:
    print(command)

cluster_plan = {
    "location": location_name,
    "pipeline": "wflow_dynamic_handoff_then_sfincs",
    "events_in_joint_worklist": int(len(pipeline_readiness)),
    "accepted_before_cluster_run": int(len(pipeline_accepted)),
    "events_without_current_handoff": int(len(pipeline_blocked)),
    "joint_worklist_csv": str(joint_worklist_csv),
    "readiness_worklist_csv": str(readiness_worklist_csv),
    "blocked_worklist_csv": str(blocked_worklist_csv),
    "accepted_worklist_csv": str(accepted_worklist_csv),
    "coupled_pipeline_slurm": str(repo_root / coupled_pipeline_slurm),
    "force_wflow": True,
    "overwrite_meteo": True,
    "commands": coupled_pipeline_commands,
}
cluster_plan_path = scenario_root_for_worklists / f"{location_name}_joint_wflow_sfincs_cluster_plan.json"
cluster_plan_path.write_text(json.dumps(cluster_plan, indent=2) + "\n", encoding="utf-8")

display(pd.Series(cluster_plan, name="coupled_wflow_sfincs_cluster_pipeline"))

## SFINCS Scenario Folders


In [ ]:
stage_scenarios = False  # production uses the joint cluster pipeline above
force_restage = True
scenario_limit = None
require_full_catalog_acceptance = True
scenario_catalog_path = resolve_location_path("data/event_catalog/catalog/scenario_catalog.csv")
stress_training_catalog_path = resolve_location_path("data/event_catalog/catalog/resilience_stress_training_catalog.csv")
probability_catalog_path = resolve_location_path("data/event_catalog/catalog/probability_catalog.csv")

handoff_readiness = dynamic_handoff_readiness_table(
    runtime_config,
    location_root,
    catalog_path=scenario_catalog_path,
    limit=scenario_limit,
)
accepted_handoffs = handoff_readiness.loc[handoff_readiness["status"].eq("accepted")].copy()
blocked_handoffs = handoff_readiness.loc[handoff_readiness["status"].ne("accepted")].copy()

readiness_dir = resolve_location_path(sfincs_config["paths"]["scenarios_root"])
readiness_dir.mkdir(parents=True, exist_ok=True)
readiness_csv = readiness_dir / f"{location_name}_dynamic_handoff_readiness.csv"
blocked_csv = readiness_dir / f"{location_name}_blocked_dynamic_handoffs.csv"
accepted_csv = readiness_dir / f"{location_name}_accepted_dynamic_handoffs.csv"
handoff_readiness.to_csv(readiness_csv, index=False)
blocked_handoffs.to_csv(blocked_csv, index=False)
accepted_handoffs.to_csv(accepted_csv, index=False)

display(pd.Series({
    "accepted_dynamic_handoffs": len(accepted_handoffs),
    "blocked_events": len(blocked_handoffs),
    "catalog_events_checked": len(handoff_readiness),
    "readiness_csv": str(readiness_csv),
    "blocked_csv": str(blocked_csv),
    "accepted_csv": str(accepted_csv),
}, name="dynamic_handoff_readiness"))
display(accepted_handoffs[["event_id", "sfincs_discharge_forcing", "acceptance"]])
display(blocked_handoffs.head(20))

accepted_event_ids = accepted_dynamic_handoff_event_ids(handoff_readiness)
if stage_scenarios:
    if require_full_catalog_acceptance and not blocked_handoffs.empty:
        raise RuntimeError(
            f"Only {len(accepted_handoffs)} of {len(handoff_readiness)} catalog events have accepted dynamic Wflow handoffs. "
            "Submit the joint Wflow->SFINCS cluster pipeline from the previous cell for the full joint worklist, "
            "then rerun this optional local staging cell only if you need local/prepared scenario folders."
        )
    if not accepted_event_ids:
        raise RuntimeError("No accepted dynamic Wflow handoffs are available. Run 04/b_prepare_wflow_dynamic_handoff.ipynb first.")
    scenario_report = stage_inland_coupled_scenarios(
        runtime_config,
        {"location_root": location_root},
        catalog_path=scenario_catalog_path,
        event_ids=accepted_event_ids,
        force=force_restage,
    )
    forcing_report = stage_inland_coupled_scenario_forcing(
        runtime_config,
        {"location_root": location_root},
        scenario_report=scenario_report,
    )
    display(scenario_report[["event_id", "sfincs_domain_id", "scenario_status", "run_root", "wflow_discharge_forcing"]])
    display(forcing_report[["event_id", "sfincs_domain_id", "status", "n_src", "direct_rainfall_enabled", "netamprfile", "inifile"]])
else:
    scenarios_root = resolve_location_path(sfincs_config["paths"]["scenarios_root"])
    scenario_count = len(pd.read_csv(scenario_catalog_path)) if scenario_catalog_path.exists() else 0
    stress_training_count = len(pd.read_csv(stress_training_catalog_path)) if stress_training_catalog_path.exists() else 0
    scenario_report = pd.Series({
        "stage_scenarios": stage_scenarios,
        "accepted_dynamic_handoffs": len(accepted_event_ids),
        "blocked_events_for_joint_pipeline": len(blocked_handoffs),
        "scenario_catalog_csv": str(scenario_catalog_path),
        "scenario_count": scenario_count,
        "stress_training_catalog_csv": str(stress_training_catalog_path),
        "stress_training_count": stress_training_count,
        "probability_catalog_csv": str(probability_catalog_path),
        "target_root": str(scenarios_root),
        "status": "joint cluster pipeline prepared; local SFINCS staging skipped",
        "next_step": "Submit cluster/run_wflow_sfincs_dsai_inland_coupled.slurm using the printed command above.",
    })
    forcing_report = pd.DataFrame()
    display(scenario_report)

## Scenario Folder Audit


In [ ]:
if accepted_event_ids:
    scenario_audit = audit_inland_coupled_batch_readiness(
        runtime_config,
        {"location_root": location_root},
        catalog_path=scenario_catalog_path,
        event_ids=accepted_event_ids,
    )
    display(scenario_audit["status"].value_counts().rename("readiness_check_count"))
    display(scenario_audit.loc[scenario_audit["status"].ne("passed")].head(30))

    failed_checks = scenario_audit.loc[scenario_audit["status"].ne("passed")]
    if not failed_checks.empty and stage_scenarios:
        raise RuntimeError("SFINCS scenario staging is not end-to-end ready; resolve failed readiness checks before cluster submission.")
else:
    scenario_audit = pd.DataFrame()
    display(pd.Series({"status": "no accepted events to audit yet"}, name="scenario_folder_audit"))

exists_table(location_root, {
    "scenarios root": sfincs_config["paths"]["scenarios_root"],
    "scenario catalog": f"{sfincs_config['paths']['scenarios_root']}/scenario_catalog.csv",
    "scenario forcing report": f"{sfincs_config['paths']['scenarios_root']}/scenario_forcing_report.csv",
    "run stage root": sfincs_config["paths"]["run_root"],
    "run outputs root": sfincs_config["paths"]["storage_root"],
    "stats root": sfincs_config["paths"]["stats_root"],
})


## Run on the Cluster - End-to-End

The joint Wflow-SFINCS pipeline (`cluster/run_wflow_sfincs_dsai_inland_coupled.slurm`) is the production path. Each array task runs one event in order: **Wflow dynamic handoff -> native SFINCS staging -> SFINCS run**. The command printed above uses `FORCE_WFLOW=1`, so accepted handoff artifacts are recomputed rather than reused.

**1. Push inputs to the cluster:**

```bash
SYNC_LOCATION=greensboro cluster/sync_to_dsai.sh --run-inputs-only
```

**2. Debug one event:**

```bash
mkdir -p runs
sbatch --array=0-0 --export=ALL,LOCATION=greensboro,EVENT_IDS=<event_id>,FORCE_RERUN=1,FORCE_WFLOW=1,OVERWRITE_METEO=1,KEEP_STAGE=1 cluster/run_wflow_sfincs_dsai_inland_coupled.slurm
```

**3. Run the full resilience catalog:**

```bash
sbatch --array=0-31%8 --export=ALL,LOCATION=greensboro,WORKLIST=locations/greensboro/data/sfincs/scenarios/greensboro_joint_wflow_sfincs_worklist.csv,FORCE_RERUN=1,FORCE_WFLOW=1,OVERWRITE_METEO=1 cluster/run_wflow_sfincs_dsai_inland_coupled.slurm
```

Outputs are written under `locations/greensboro/data/sfincs/run_outputs` on the cluster. The optional SFINCS scenario-folder cells above are for accepted-only local inspection; they are not the production coupled run path.